#### Training:   Jan 1 2024 → Sep 30 2025  (21 months, 638 days)
#### Validation: Oct 1 2025 → Dec 31 2025  (3 months,  92 days)
#### Predict:    Jan 1 2026 → May 31 2026  (151 days)

In [1]:
import polars as pl
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

# ── paths ──────────────────────────────────────────────────────────
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR    = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── plotly defaults ────────────────────────────────────────────────
PLOT_CFG = dict(
    width=1200, height=700,
    margin=dict(l=50, r=50, t=50, b=120),
    legend=dict(
        orientation="h", yanchor="top",
        y=-0.15, xanchor="center", x=0.5,
        font=dict(size=10)
    )
)

# ── load ───────────────────────────────────────────────────────────
df = pl.read_parquet(PROCESSED_DIR / "avl_fleet_daily.parquet")

# ── split boundaries ──────────────────────────────────────────────
TRAIN_END = "2025-09-30"
VAL_START = "2025-10-01"
VAL_END   = "2025-12-31"
PRED_START = "2026-01-01"
PRED_END   = "2026-05-31"

# ── create splits ─────────────────────────────────────────────────
df_train = df.filter(pl.col("DATE_DAY") <= pl.lit(TRAIN_END).str.to_date())
df_val   = df.filter(
    (pl.col("DATE_DAY") >= pl.lit(VAL_START).str.to_date()) &
    (pl.col("DATE_DAY") <= pl.lit(VAL_END).str.to_date())
)

# ── sanity checks ─────────────────────────────────────────────────
print(f"Train: {df_train['DATE_DAY'].min()} → {df_train['DATE_DAY'].max()} ({len(df_train)} days)")
print(f"Val:   {df_val['DATE_DAY'].min()} → {df_val['DATE_DAY'].max()} ({len(df_val)} days)")
print(f"\nTrain + Val = {len(df_train) + len(df_val)} days (expect 731)")

# ── check: no date gaps in training data ──────────────────────────
# gaps in training data can silently break time series models
date_diffs = (
    df_train
    .with_columns(
        pl.col("DATE_DAY").diff().alias("gap")
    )
    .filter(pl.col("gap") > pl.duration(days=1))
)
print(f"\nGaps in training data: {len(date_diffs)} (expect 0)")
if len(date_diffs) > 0:
    print("⚠️  Gaps found on these dates:")
    print(date_diffs.select(["DATE_DAY", "gap"]))

Train: 2024-01-01 → 2025-09-30 (639 days)
Val:   2025-10-01 → 2025-12-31 (92 days)

Train + Val = 731 days (expect 731)

Gaps in training data: 0 (expect 0)


In [2]:
# ── Section 2: Seasonal Mean Baseline ────────────────────────────

rate_cols = ["aog_rate", "smx_rate", "umx_rate", "cna_rate"]

# compute monthly mean for each rate from TRAINING data only
monthly_means = (
    df_train
    .with_columns(pl.col("DATE_DAY").dt.month().alias("month"))
    .group_by("month")
    .agg([pl.col(c).mean().alias(f"{c}_baseline") for c in rate_cols])
    .sort("month")
)

print("Monthly mean rates (training data):")
print(monthly_means)

# ── apply baseline to validation set ─────────────────────────────
df_val_baseline = (
    df_val
    .with_columns(pl.col("DATE_DAY").dt.month().alias("month"))
    .join(monthly_means, on="month", how="left")
)

# ── compute RMSE and MAE per component ───────────────────────────
# Why both RMSE and MAE?
# MAE = average absolute error (treats all errors equally)
# RMSE = penalises large errors more (a 20% spike matters more than two 10% errors)
# For supply forecasting, large errors matter more → RMSE is primary metric

results_baseline = {}
for col in rate_cols:
    actual    = df_val_baseline[col].to_numpy()
    predicted = df_val_baseline[f"{col}_baseline"].to_numpy()
    mae  = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    results_baseline[col] = {"MAE": round(mae, 6), "RMSE": round(rmse, 6)}
    print(f"{col:12s} → RMSE: {rmse:.6f}  |  MAE: {mae:.6f}")

# ── plot baseline predictions vs actuals ─────────────────────────
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    f"AOG — Baseline vs Actual",
    f"SMX — Baseline vs Actual",
    f"UMX — Baseline vs Actual",
    f"CNA — Baseline vs Actual"
])

colors_actual   = ["#dc2626","#2563eb","#d97706","#16a34a"]
colors_baseline = ["#fca5a5","#93c5fd","#fde68a","#86efac"]

for idx, col in enumerate(rate_cols):
    row, col_pos = divmod(idx, 2)
    dates = df_val_baseline["DATE_DAY"].to_list()

    fig.add_trace(go.Scatter(
        x=dates, y=df_val_baseline[col].to_list(),
        mode="lines", name=f"{col} actual",
        line=dict(color=colors_actual[idx], width=1.5)
    ), row=row+1, col=col_pos+1)

    fig.add_trace(go.Scatter(
        x=dates, y=df_val_baseline[f"{col}_baseline"].to_list(),
        mode="lines", name=f"{col} baseline",
        line=dict(color=colors_baseline[idx], width=2, dash="dash")
    ), row=row+1, col=col_pos+1)

fig.update_layout(
    title="Seasonal Mean Baseline vs Actuals — Validation Period (Oct–Dec 2025)",
    height=900, width=1200,
    margin=dict(l=50, r=50, t=80, b=80)
)
fig.show()

Monthly mean rates (training data):
shape: (12, 5)
┌───────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┐
│ month ┆ aog_rate_baseline ┆ smx_rate_baseline ┆ umx_rate_baseline ┆ cna_rate_baseline │
│ ---   ┆ ---               ┆ ---               ┆ ---               ┆ ---               │
│ i8    ┆ f64               ┆ f64               ┆ f64               ┆ f64               │
╞═══════╪═══════════════════╪═══════════════════╪═══════════════════╪═══════════════════╡
│ 1     ┆ 0.034509          ┆ 0.087022          ┆ 0.164666          ┆ 0.029257          │
│ 2     ┆ 0.042024          ┆ 0.107711          ┆ 0.140759          ┆ 0.031008          │
│ 3     ┆ 0.034509          ┆ 0.092648          ┆ 0.15829           ┆ 0.012378          │
│ 4     ┆ 0.02907           ┆ 0.122481          ┆ 0.121318          ┆ 0.014729          │
│ 5     ┆ 0.036384          ┆ 0.124156          ┆ 0.159415          ┆ 0.006377          │
│ …     ┆ …                 ┆ …                 ┆

In [3]:
# ── Section 3: AOG Model — SARIMA(1,0,0) ──────────────────────────

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings("ignore")   # statsmodels is verbose

# ── Step 1: Stationarity test (ADF test) ──────────────────────────
# Before fitting any ARIMA, confirm our d=0 assumption for AOG
# ADF test null hypothesis: series HAS a unit root (non-stationary)
# If p-value < 0.05 → reject null → series IS stationary → d=0 correct

aog_train = df_train["aog_rate"].to_numpy()

adf_result = adfuller(aog_train, autolag="AIC")
print("── ADF Stationarity Test: AOG ──")
print(f"  ADF Statistic : {adf_result[0]:.4f}")
print(f"  p-value       : {adf_result[1]:.4f}")
print(f"  Verdict       : {'STATIONARY ✅ (d=0)' if adf_result[1] < 0.05 else 'NON-STATIONARY ⚠️ (consider d=1)'}")

# ── Step 2: Fit SARIMA(1,0,0) on training data ────────────────────
model_aog = ARIMA(
    aog_train,
    order=(1, 0, 0),   # p=1, d=0, q=0
)
fitted_aog = model_aog.fit()

print("\n── Model Summary (key lines) ──")
print(f"  AR coefficient (phi) : {fitted_aog.params[1]:.4f}")
print(f"  Intercept            : {fitted_aog.params[0]:.4f}")
print(f"  AIC                  : {fitted_aog.aic:.2f}")
print(f"  BIC                  : {fitted_aog.bic:.2f}")

# ── Step 3: Forecast the validation period (92 days) ──────────────
# get_forecast gives us both point forecast + confidence intervals
forecast_obj  = fitted_aog.get_forecast(steps=len(df_val))
aog_val_pred  = forecast_obj.predicted_mean
aog_val_ci    = forecast_obj.conf_int(alpha=0.2)  # 80% CI

# ── Step 4: Evaluate vs baseline ──────────────────────────────────
aog_actual = df_val["aog_rate"].to_numpy()

rmse_sarima   = np.sqrt(np.mean((aog_actual - aog_val_pred) ** 2))
mae_sarima    = np.mean(np.abs(aog_actual - aog_val_pred))
rmse_baseline = results_baseline["aog_rate"]["RMSE"]
mae_baseline  = results_baseline["aog_rate"]["MAE"]

print("\n── AOG Evaluation: SARIMA(1,0,0) vs Baseline ──")
print(f"{'':20s}  {'RMSE':>10}  {'MAE':>10}")
print(f"{'Baseline':20s}  {rmse_baseline:>10.6f}  {mae_baseline:>10.6f}")
print(f"{'SARIMA(1,0,0)':20s}  {rmse_sarima:>10.6f}  {mae_sarima:>10.6f}")
improvement = (rmse_baseline - rmse_sarima) / rmse_baseline * 100
print(f"\nRMSE improvement over baseline: {improvement:.1f}%")

# ── Step 5: Plot forecast vs actuals ──────────────────────────────
val_dates = df_val["DATE_DAY"].to_list()

fig = go.Figure()

# actuals
fig.add_trace(go.Scatter(
    x=val_dates, y=aog_actual,
    mode="lines", name="AOG Actual",
    line=dict(color="#dc2626", width=1.5)
))

# SARIMA forecast
fig.add_trace(go.Scatter(
    x=val_dates, y=aog_val_pred,
    mode="lines", name="SARIMA(1,0,0) Forecast",
    line=dict(color="#1d4ed8", width=2)
))

# 80% confidence interval
fig.add_trace(go.Scatter(
    x=val_dates + val_dates[::-1],
    y=list(aog_val_ci[:, 1]) + list(aog_val_ci[:, 0])[::-1],
    fill="toself", fillcolor="rgba(29,78,216,0.15)",
    line=dict(width=0), name="80% CI"
))

# baseline
baseline_vals = df_val.with_columns(
    pl.col("DATE_DAY").dt.month().alias("month")
).join(monthly_means, on="month").select("aog_rate_baseline").to_series().to_list()
fig.add_trace(go.Scatter(
    x=val_dates, y=baseline_vals,
    mode="lines", name="Seasonal Mean Baseline",
    line=dict(color="#f97316", width=2, dash="dash")
))

fig.update_layout(
    title="AOG Rate — SARIMA(1,0,0) vs Baseline vs Actual (Oct–Dec 2025)",
    xaxis_title="Date", yaxis_title="AOG Rate",
    **PLOT_CFG
)
fig.show()

── ADF Stationarity Test: AOG ──
  ADF Statistic : -15.6444
  p-value       : 0.0000
  Verdict       : STATIONARY ✅ (d=0)

── Model Summary (key lines) ──
  AR coefficient (phi) : 0.3547
  Intercept            : 0.0367
  AIC                  : -2833.80
  BIC                  : -2820.42

── AOG Evaluation: SARIMA(1,0,0) vs Baseline ──
                            RMSE         MAE
Baseline                0.047174    0.033154
SARIMA(1,0,0)           0.047005    0.033080

RMSE improvement over baseline: 0.4%


Prophet expects a dataframe with exactly two columns:
  ds  →  datestamp (your DATE_DAY)
  y   →  the value to forecast (the rate)

You tell it what seasonality to model:
  yearly_seasonality=True   → captures the annual wave (UMX's big signal)
  weekly_seasonality=False  → we saw day-of-week is flat → turn off
  daily_seasonality=False   → we have daily data but no sub-daily pattern

It returns:
  yhat        →  point forecast
  yhat_lower  →  lower confidence bound
  yhat_upper  →  upper confidence bound

In [4]:
from prophet import Prophet

def fit_evaluate_prophet(
    df_train, df_val, rate_col, monthly_means, results_baseline,
    yearly_seasonality=True, weekly_seasonality=False
):
    """
    Fits Prophet on rate_col, evaluates on df_val, returns metrics + forecast df.
    Encapsulated as a function so we call it identically for SMX and UMX.
    """

    # ── Prophet requires columns named 'ds' and 'y' ──────────────
    df_prophet_train = (
        df_train
        .select(["DATE_DAY", rate_col])
        .rename({"DATE_DAY": "ds", rate_col: "y"})
        .to_pandas()
    )

    # ── ADF stationarity check ────────────────────────────────────
    adf_result = adfuller(df_prophet_train["y"].values, autolag="AIC")
    print(f"\n── ADF Test: {rate_col} ──")
    print(f"  p-value : {adf_result[1]:.4f}")
    print(f"  Verdict : {'STATIONARY ✅' if adf_result[1] < 0.05 else 'NON-STATIONARY ⚠️'}")

    # ── fit Prophet ───────────────────────────────────────────────
    model = Prophet(
        yearly_seasonality=yearly_seasonality,
        weekly_seasonality=weekly_seasonality,
        daily_seasonality=False,
        interval_width=0.80,         # 80% CI
        changepoint_prior_scale=0.05 # how flexible the trend is
                                     # 0.05 = moderately flexible
                                     # lower = smoother trend
    )
    model.fit(df_prophet_train)

    # ── build future dataframe for val + Jan-May 2026 ─────────────────
    # we predict val + prediction horizon together in one go
    future = model.make_future_dataframe(
        periods=len(df_val) + 151,  # 92 val days already in future + 151 for Jan-May 2026
        freq="D"
    )
    forecast = model.predict(future)

    # ── slice validation period ───────────────────────────────────
    val_dates = df_val["DATE_DAY"].to_pandas()
    fc_val = forecast[forecast["ds"].isin(val_dates)].reset_index(drop=True)

    # ── evaluate ──────────────────────────────────────────────────
    actual    = df_val[rate_col].to_numpy()
    predicted = fc_val["yhat"].values
    # clip predictions to [0, 1] — rates can't be negative
    predicted = np.clip(predicted, 0, 1)

    rmse_prophet   = np.sqrt(np.mean((actual - predicted) ** 2))
    mae_prophet    = np.mean(np.abs(actual - predicted))
    rmse_baseline  = results_baseline[rate_col]["RMSE"]
    mae_baseline   = results_baseline[rate_col]["MAE"]
    improvement    = (rmse_baseline - rmse_prophet) / rmse_baseline * 100

    print(f"\n── {rate_col.upper()} Evaluation ──")
    print(f"{'':20s}  {'RMSE':>10}  {'MAE':>10}")
    print(f"{'Baseline':20s}  {rmse_baseline:>10.6f}  {mae_baseline:>10.6f}")
    print(f"{'Prophet':20s}  {rmse_prophet:>10.6f}  {mae_prophet:>10.6f}")
    print(f"RMSE improvement: {improvement:.1f}%")

    return model, forecast, fc_val, predicted, improvement


# ── run for SMX ──────────────────────────────────────────────────
model_smx, fc_smx_full, fc_smx_val, smx_pred, smx_improvement = \
    fit_evaluate_prophet(df_train, df_val, "smx_rate",
                         monthly_means, results_baseline)

# ── run for UMX ──────────────────────────────────────────────────
model_umx, fc_umx_full, fc_umx_val, umx_pred, umx_improvement = \
    fit_evaluate_prophet(df_train, df_val, "umx_rate",
                         monthly_means, results_baseline)

15:31:55 - cmdstanpy - INFO - Chain [1] start processing
15:31:55 - cmdstanpy - INFO - Chain [1] done processing
15:31:55 - cmdstanpy - INFO - Chain [1] start processing



── ADF Test: smx_rate ──
  p-value : 0.0000
  Verdict : STATIONARY ✅

── SMX_RATE Evaluation ──
                            RMSE         MAE
Baseline                0.054457    0.044818
Prophet                 0.044455    0.035615
RMSE improvement: 18.4%

── ADF Test: umx_rate ──
  p-value : 0.0000
  Verdict : STATIONARY ✅


15:31:55 - cmdstanpy - INFO - Chain [1] done processing



── UMX_RATE Evaluation ──
                            RMSE         MAE
Baseline                0.050433    0.040540
Prophet                 0.057141    0.044021
RMSE improvement: -13.3%


In [5]:
# ── Section 6: CNA constant + Final comparison ───────────────────

# CNA: constant = mean of training data
cna_constant = df_train["cna_rate"].mean()
cna_actual   = df_val["cna_rate"].to_numpy()
cna_pred     = np.full(len(df_val), cna_constant)
rmse_cna     = np.sqrt(np.mean((cna_actual - cna_pred) ** 2))
mae_cna      = np.mean(np.abs(cna_actual - cna_pred))

print(f"\n── CNA Constant Rate ──")
print(f"  Constant value : {cna_constant:.6f}  ({cna_constant*100:.2f}%)")
print(f"  RMSE           : {rmse_cna:.6f}")
print(f"  Baseline RMSE  : {results_baseline['cna_rate']['RMSE']:.6f}")

# ── Final model decision table ────────────────────────────────────
print("\n" + "="*65)
print("FINAL MODEL DECISIONS — AVAILABILITY COMPONENTS")
print("="*65)
print(f"{'Component':<12} {'Model':>20} {'RMSE':>10} {'vs Baseline':>14}")
print("-"*65)
print(f"{'AOG':<12} {'Seasonal Mean':>20} {results_baseline['aog_rate']['RMSE']:>10.6f} {'(baseline = model)':>14}")
print(f"{'SMX':<12} {'Prophet':>20} {np.sqrt(np.mean((df_val['smx_rate'].to_numpy()-smx_pred)**2)):>10.6f} {f'{smx_improvement:+.1f}%':>14}")
print(f"{'UMX':<12} {'Prophet':>20} {np.sqrt(np.mean((df_val['umx_rate'].to_numpy()-umx_pred)**2)):>10.6f} {f'{umx_improvement:+.1f}%':>14}")
print(f"{'CNA':<12} {'Constant mean':>20} {rmse_cna:>10.6f} {'(confirmed)':>14}")

# ── 4-panel final comparison plot ────────────────────────────────
val_dates   = df_val["DATE_DAY"].to_list()
final_preds = {
    "aog_rate": np.array(
        df_val.with_columns(pl.col("DATE_DAY").dt.month().alias("month"))
        .join(monthly_means, on="month")
        .select("aog_rate_baseline").to_series().to_list()
    ),
    "smx_rate": smx_pred,
    "umx_rate": umx_pred,
    "cna_rate": cna_pred
}
model_names = {
    "aog_rate": "Seasonal Mean Baseline",
    "smx_rate": "Prophet",
    "umx_rate": "Prophet",
    "cna_rate": "Constant Mean"
}
colors_model  = ["#f97316","#2563eb","#d97706","#16a34a"]
colors_actual = ["#dc2626","#93c5fd","#fed7aa","#86efac"]

fig = make_subplots(rows=2, cols=2, subplot_titles=[
    "AOG — Final Model vs Actual",
    "SMX — Prophet vs Actual",
    "UMX — Prophet vs Actual",
    "CNA — Constant Mean vs Actual"
])

for idx, col in enumerate(["aog_rate","smx_rate","umx_rate","cna_rate"]):
    row, col_pos = divmod(idx, 2)
    actual = df_val[col].to_list()

    fig.add_trace(go.Scatter(
        x=val_dates, y=actual,
        mode="lines", name=f"{col} actual",
        line=dict(color=colors_actual[idx], width=1.5)
    ), row=row+1, col=col_pos+1)

    fig.add_trace(go.Scatter(
        x=val_dates, y=final_preds[col].tolist(),
        mode="lines", name=model_names[col],
        line=dict(color=colors_model[idx], width=2)
    ), row=row+1, col=col_pos+1)

fig.update_layout(
    title="Final Availability Models — All 4 Components vs Actuals",
    height=900, width=1200,
    margin=dict(l=50, r=50, t=80, b=80),
    showlegend=False
)
fig.show()


── CNA Constant Rate ──
  Constant value : 0.018396  (1.84%)
  RMSE           : 0.017446
  Baseline RMSE  : 0.016823

FINAL MODEL DECISIONS — AVAILABILITY COMPONENTS
Component                   Model       RMSE    vs Baseline
-----------------------------------------------------------------
AOG                 Seasonal Mean   0.047174 (baseline = model)
SMX                       Prophet   0.044455         +18.4%
UMX                       Prophet   0.057141         -13.3%
CNA                 Constant mean   0.017446    (confirmed)


In [6]:
# Quick retry — add weekly seasonality since ACF confirmed it for UMX
model_umx_v2, fc_umx_v2_full, fc_umx_v2_val, umx_pred_v2, umx_improvement_v2 = \
    fit_evaluate_prophet(df_train, df_val, "umx_rate",
                         monthly_means, results_baseline,
                         yearly_seasonality=True,
                         weekly_seasonality=True)   # ← changed

15:31:56 - cmdstanpy - INFO - Chain [1] start processing
15:31:56 - cmdstanpy - INFO - Chain [1] done processing



── ADF Test: umx_rate ──
  p-value : 0.0000
  Verdict : STATIONARY ✅

── UMX_RATE Evaluation ──
                            RMSE         MAE
Baseline                0.050433    0.040540
Prophet                 0.055936    0.043344
RMSE improvement: -10.9%


In [7]:
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
# Note: proper TBATS needs tbats package
# uv add tbats
from tbats import TBATS as TBATSModel

umx_train_values = df_train["umx_rate"].to_numpy()

# TBATS with two seasonal periods: weekly (7) and annual (365.25)
tbats_estimator = TBATSModel(
    seasonal_periods=[7],  # weekly + annual simultaneously
    use_box_cox=True,              # stabilise variance first (rates are bounded)
    use_trend=True,
    use_damped_trend=True          # prevents trend from extrapolating wildly
)
tbats_fitted = tbats_estimator.fit(umx_train_values)

# forecast 92 val days
umx_tbats_pred = tbats_fitted.forecast(steps=len(df_val))
umx_tbats_pred = np.clip(umx_tbats_pred, 0, 1)

# evaluate
umx_actual    = df_val["umx_rate"].to_numpy()
rmse_tbats    = np.sqrt(np.mean((umx_actual - umx_tbats_pred)**2))
rmse_baseline = results_baseline["umx_rate"]["RMSE"]
tbats_improvement = (rmse_baseline - rmse_tbats) / rmse_baseline * 100

print("── UMX: All Models Compared ──")
print(f"{'Model':<25} {'RMSE':>10}  {'vs Baseline':>14}")
print(f"{'Baseline':.<25} {rmse_baseline:>10.6f}  {'(benchmark)':>14}")
print(f"{'Prophet (annual only)':<25} {0.057141:>10.6f}  {'-13.3%':>14}")
print(f"{'Prophet (annual+weekly)':<25} {'run above':>10}  {'TBD':>14}")
print(f"{'TBATS (7 + 365.25)':<25} {rmse_tbats:>10.6f}  {f'{tbats_improvement:+.1f}%':>14}")

── UMX: All Models Compared ──
Model                           RMSE     vs Baseline
Baseline.................   0.050433     (benchmark)
Prophet (annual only)       0.057141          -13.3%
Prophet (annual+weekly)    run above             TBD
TBATS (7 + 365.25)          0.057299          -13.6%


In [8]:
import pickle

# ── save model artifacts ──────────────────────────────────────────
artifacts = {
    "smx_prophet_model"   : model_smx,
    "monthly_means"       : monthly_means.to_pandas(),
    "cna_constant_rate"   : float(cna_constant),
    "baseline_metrics"    : results_baseline,
    "train_end"           : TRAIN_END,
}
with open(MODELS_DIR / "availability_models.pkl", "wb") as f:
    pickle.dump(artifacts, f)

# ── generate Jan-May 2026 forecasts for all 4 components ─────────────
pred_dates = pd.date_range(PRED_START, PRED_END, freq="D")

# SMX → slice from Prophet's already-computed future forecast
fc_pred_smx = (
    fc_smx_full[fc_smx_full["ds"].isin(pred_dates)]
    .reset_index(drop=True)
)
smx_pred = np.clip(fc_pred_smx["yhat"].values, 0, 1)

# AOG + UMX + CNA → seasonal mean per month
df_pred_dates = pl.DataFrame({
    "date": pred_dates,
    "month": [d.month for d in pred_dates]
})
df_pred_seasonal = df_pred_dates.join(monthly_means, on="month", how="left")

aog_pred = df_pred_seasonal["aog_rate_baseline"].to_numpy()
umx_pred = df_pred_seasonal["umx_rate_baseline"].to_numpy()
cna_pred = np.full(len(pred_dates), float(cna_constant))

# ── build Jan-May 2026 availability forecast dataframe ───────────────
num_tails_forecast = 43  # stable fleet size confirmed in EDA

df_pred_avail = pl.DataFrame({
    "date"           : list(pred_dates),
    "num_tails"      : [num_tails_forecast] * len(pred_dates),
    "aog_rate_fc"    : aog_pred.tolist(),
    "smx_rate_fc"    : smx_pred.tolist(),
    "umx_rate_fc"    : umx_pred.tolist(),
    "cna_rate_fc"    : cna_pred.tolist(),
}).with_columns([
    (
        pl.col("num_tails") * (
            1
            - pl.col("aog_rate_fc")
            - pl.col("smx_rate_fc")
            - pl.col("umx_rate_fc")
            - pl.col("cna_rate_fc")
        )
    ).alias("delivered_tails_fc"),
]).with_columns(
    (pl.col("delivered_tails_fc") * 1.06).alias("raw_leg_supply_fc")
)

print("── Jan-May 2026 Availability Forecast ──")
print(df_pred_avail.select([
    "date","num_tails","delivered_tails_fc","raw_leg_supply_fc"
]))
print(f"\nMean delivered tails (Jan-May 2026): {df_pred_avail['delivered_tails_fc'].mean():.2f}")
print(f"Mean raw leg supply  (Jan-May 2026): {df_pred_avail['raw_leg_supply_fc'].mean():.2f}")

# save for downstream supply formula
df_pred_avail.write_parquet(
    PROCESSED_DIR / "jan2026_may2026_availability_forecast.parquet"
)
print("\n✅ Saved → data/processed/jan2026_may2026_availability_forecast.parquet")

── Jan-May 2026 Availability Forecast ──
shape: (151, 4)
┌─────────────────────┬───────────┬────────────────────┬───────────────────┐
│ date                ┆ num_tails ┆ delivered_tails_fc ┆ raw_leg_supply_fc │
│ ---                 ┆ ---       ┆ ---                ┆ ---               │
│ datetime[μs]        ┆ i64       ┆ f64                ┆ f64               │
╞═════════════════════╪═══════════╪════════════════════╪═══════════════════╡
│ 2026-01-01 00:00:00 ┆ 43        ┆ 30.620102          ┆ 32.457309         │
│ 2026-01-02 00:00:00 ┆ 43        ┆ 30.678266          ┆ 32.518962         │
│ 2026-01-03 00:00:00 ┆ 43        ┆ 30.719091          ┆ 32.562237         │
│ 2026-01-04 00:00:00 ┆ 43        ┆ 30.741635          ┆ 32.586133         │
│ 2026-01-05 00:00:00 ┆ 43        ┆ 30.745301          ┆ 32.59002          │
│ …                   ┆ …         ┆ …                  ┆ …                 │
│ 2026-05-27 00:00:00 ┆ 43        ┆ 27.580207          ┆ 29.235019         │
│ 2026-05-28 00:00: